# Benchmarking Classical Machine Learning Model on AptaBench
This notebook benchmarks **traditional ML model** (LightGBM) on the AptaBench dataset, focusing on feature engineering, cross-validation, and baseline predictive performance.

## 1. Imports & Config

In [2]:
import torch
torch.__version__

'2.5.1+cu118'

In [1]:
# Add project root to sys.path so "src" is importable
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root.parent != root:
    root = root.parent

sys.path.insert(0, str(root))

In [2]:
# Fix NumPy version compatibility with RDKit
# RDKit requires NumPy < 2.0, but transformers may install NumPy 2.x
import numpy as np
if np.__version__.startswith('2.'):
    import subprocess
    import sys
    print(f"NumPy version {np.__version__} detected. Downgrading to < 2.0 for RDKit compatibility...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy<2.0", "--force-reinstall", "-q"])
    print("NumPy downgraded. Please restart the kernel and run again.")
    raise RuntimeError("NumPy downgraded. Please restart the kernel (Kernel -> Restart Kernel) and run cells again.")
else:
    print(f"NumPy version {np.__version__} is compatible with RDKit.")


NumPy version 1.26.4 is compatible with RDKit.


In [3]:
# === Imports ===
import os
import pandas as pd
from src.models.screening import screen_lgbm_optuna, load_splits
# --- Aptamer encoders ---
from src.encoders.aptamer_encoders import (
    onehot_with_type_bit,
    kmer_freq_with_type_bit,
    gena_embed,
    dnabert2_embed
)

# --- Molecule encoders ---
from src.encoders.molecule_encoders import (
    morgan_fp,
    maccs_fp,
    physchem_descriptors,
    chemberta_embed,
    unimol_embed,
    molformer_embed,
    clamp_embed
)

# Paths
DATASET_PATH = "../dataset/AptaBench_dataset_v2.csv" 
SPLITS_DIR = "../dataset/splits"

c:\Users\m19er\AptaBench\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [4]:
# Load dataset
df = pd.read_csv(DATASET_PATH)
print("Dataset loaded:", df.shape)
display(df.head())

# Check available splits
print("Available splits:", os.listdir(SPLITS_DIR))

Dataset loaded: (6413, 8)


,type,sequence,canonical_smiles,pKd_value,label,buffer,origin,source
0,DNA,GGGAGAATTCCCGCGGCAGAAGCCCACCTGGCTTTGAACTCTATGT...,Nc1c(S(=O)(=O)O)cc(Nc2ccc(Nc3nc(Cl)nc(Nc4ccccc...,4.000000,0,NaN,NaN,RSAPred
1,RNA,GGGAGAAUUCCCGCGGCGUUGGCCCAGGAUAAUAGGACGAAAUCCG...,Nc1c(S(=O)(=O)O)cc(Nc2ccc(S(=O)(=O)O)c(Nc3nc(C...,3.221849,0,NaN,NaN,RSAPred
2,RNA,GGGAAGGGAAGAAACUGCGGCUUCGGCCGGCUUCCC,Nc1ncnc2c1ncn2C1OC(COP(=O)(O)OP(=O)(O)OP(=O)(O...,5.397940,1,NaN,NaN,RSAPred
3,RNA,GGGAAGGGAAGAAACUGCGGCUUCGGCCGGCUUCCC,Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1O,8.301026,1,NaN,NaN,RSAPred
4,RNA,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(CC(O)C(O)C(O)COP(...,4.638272,1,NaN,NaN,RSAPred


Available splits: ['disjoint_aptamer.json', 'disjoint_molecule.json', 'stratified.json']


## 2. Encoder configurations

In [5]:
apt_cfgs = [
    {"name": "OneHot", "func": onehot_with_type_bit},
    {"name": "Kmer3", "func": kmer_freq_with_type_bit, "kwargs": {"k": 3}},
    {"name": "Kmer4", "func": kmer_freq_with_type_bit, "kwargs": {"k": 4}},
    {"name": "GENA", "func": gena_embed},
    {"name": "DNABERT2", "func": dnabert2_embed},
]

mol_cfgs = [
    {"name": "MorganFP", "func": morgan_fp},
    {"name": "MACCS", "func": maccs_fp},
    {"name": "PhysChem", "func": physchem_descriptors},
    {"name": "ChemBERTa", "func": chemberta_embed},
    {"name": "UniMol", "func": unimol_embed},
    {"name": "MolFormer", "func": molformer_embed},
    {"name": "CLAMP", "func": clamp_embed},
]

In [6]:
  from src.models.screening import precompute_and_save_embeddings
  precompute_and_save_embeddings(df, apt_cfgs, mol_cfgs)

Dataset hash: 556cdb84df79cb8068f04358678fd4df
Cache directory: notebooks\data\embeddings
Loading cached aptamer embeddings...
  ✓ Loaded OneHot: shape (6413, 865)
  ✓ Loaded Kmer3: shape (6413, 65)
  ✓ Loaded Kmer4: shape (6413, 257)
  ✓ Loaded GENA: shape (6413, 769)
  ✓ Loaded DNABERT2: shape (6413, 769)
Loading cached molecule embeddings...
  ✓ Loaded MorganFP: shape (6413, 2048)
  ✓ Loaded MACCS: shape (6413, 167)
  ✓ Loaded PhysChem: shape (6413, 196)
  ✓ Loaded ChemBERTa: shape (6413, 768)
  ✓ Loaded UniMol: shape (6413, 512)
  ✓ Loaded MolFormer: shape (6413, 768)
  ✓ Loaded CLAMP: shape (6413, 768)


({'OneHot': array([[0., 0., 1., ..., 0., 0., 0.],
         [0., 0., 1., ..., 0., 0., 1.],
         [0., 0., 1., ..., 0., 0., 1.],
         ...,
         [0., 0., 1., ..., 0., 0., 0.],
         [0., 0., 1., ..., 0., 0., 0.],
         [0., 0., 1., ..., 0., 0., 0.]]),
  'Kmer3': array([[0.02307692, 0.03076923, 0.02307692, ..., 0.01538462, 0.00769231,
          0.        ],
         [0.03100775, 0.03100775, 0.00775194, ..., 0.00775194, 0.        ,
          1.        ],
         [0.02941176, 0.02941176, 0.05882353, ..., 0.        , 0.        ,
          1.        ],
         ...,
         [0.        , 0.        , 0.02272727, ..., 0.        , 0.        ,
          0.        ],
         [0.        , 0.        , 0.02272727, ..., 0.04545455, 0.        ,
          0.        ],
         [0.        , 0.        , 0.        , ..., 0.04545455, 0.04545455,
          0.        ]]),
  'Kmer4': array([[0.00775194, 0.01550388, 0.        , ..., 0.00775194, 0.        ,
          0.        ],
         [0.01

In [6]:
# Регрессия (использует загруженные эмбеддинги из ячейки 6)
from src.models.screening import screen_lgbm_regression_simple, load_embeddings
import gc

print(f"Regression: {df[df['pKd_value'].notna()].shape[0]}/{len(df)} samples")

# Загружаем эмбеддинги для ПОЛНОГО датасета
result = load_embeddings(df, apt_cfgs, mol_cfgs, embeddings_dir="notebooks/data/embeddings")
if result is None:
    print("❌ Run cell 6 first!")
else:
    apt_map, mol_map = result
    print(f"✓ Loaded: {len(apt_map)} apt × {len(mol_map)} mol")
    
    gc.collect()
    results_reg = screen_lgbm_regression_simple(
        df, apt_map, mol_map,
        split_modes=["stratified", "disjoint_molecule", "disjoint_aptamer"],
        splits_dir="../dataset/splits",
    )
    
    results_reg.to_csv('regression_results.csv', index=False)
    print("\n✓ Saved to regression_results.csv")


Regression: 2223/6413 samples
✓ Loaded: 5 apt × 7 mol
Regression dataset: 2223 samples with valid pKd (from 6413 total)

Processing: stratified


Aptamer encoders:   0%|          | 0/5 [00:00<?, ?it/s]


IndexError: index 2224 is out of bounds for axis 0 with size 2223

In [ ]:
# Top 5 по каждому split
for split in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    print(f"\n{split.upper()} - Top 5 (lowest MAE):")
    top5 = results_reg[results_reg['split'] == split].nsmallest(5, 'MAE mean')
    print(top5[['aptamer_encoder', 'molecule_encoder', 'MAE mean', 'RMSE mean', 'R2 mean']].to_string(index=False))



In [ ]:
# Pivot таблицы с heatmap
for split in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    print(f"\n{split.upper()} - MAE:")
    pivot = results_reg[results_reg['split'] == split].pivot(
        index="aptamer_encoder", columns="molecule_encoder", values="MAE mean"
    )
    display(pivot.style.background_gradient(cmap='RdYlGn_r', axis=None).format("{:.3f}"))



In [ ]:
# Визуализация Top 10
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, split in enumerate(["stratified", "disjoint_molecule", "disjoint_aptamer"]):
    data = results_reg[results_reg['split'] == split].nsmallest(10, 'MAE mean')
    labels = [f"{row['aptamer_encoder']}+{row['molecule_encoder']}" for _, row in data.iterrows()]
    
    axes[idx].barh(range(len(data)), data['MAE mean'].values, color='steelblue')
    axes[idx].set_yticks(range(len(data)))
    axes[idx].set_yticklabels(labels, fontsize=8)
    axes[idx].set_xlabel('MAE')
    axes[idx].set_title(f'{split} (Top 10)')
    axes[idx].invert_yaxis()
    axes[idx].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
# Сравнение с классификацией (лучшая комбинация: Kmer4 + PhysChem)
clf_results = pd.read_csv('lgbm_results_screening_complete.csv')
best_apt, best_mol = "Kmer4", "PhysChem"

print(f"Best combination from classification: {best_apt} + {best_mol}\n")

for split in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    clf = clf_results[(clf_results['split']==split) & 
                      (clf_results['aptamer_encoder']==best_apt) & 
                      (clf_results['molecule_encoder']==best_mol)]
    reg = results_reg[(results_reg['split']==split) & 
                      (results_reg['aptamer_encoder']==best_apt) & 
                      (results_reg['molecule_encoder']==best_mol)]
    
    if not clf.empty and not reg.empty:
        print(f"{split}:")
        print(f"  Classification: MCC={clf['MCC mean'].values[0]:.4f}, ROC-AUC={clf['ROC-AUC mean'].values[0]:.4f}")
        print(f"  Regression:     MAE={reg['MAE mean'].values[0]:.4f}, R²={reg['R2 mean'].values[0]:.4f}\n")



In [ ]:
# Сохранение в artifacts
from pathlib import Path
import shutil

artifacts_dir = Path("../artifacts/metrics")
artifacts_dir.mkdir(parents=True, exist_ok=True)

shutil.copy('regression_results.csv', artifacts_dir / 'regression_model_results.csv')
print(f"✓ Copied to {artifacts_dir / 'regression_model_results.csv'}")



## 3. 

In [6]:
# УСКОРЕННАЯ ВЕРСИЯ С ЧЕКПОИНТАМИ И ОНЛАЙН СОХРАНЕНИЕМ
# - Чекпоинты после каждой комбинации энкодеров
# - Автоматическое продолжение с места остановки
# - Онлайн сохранение результатов после каждой итерации
# - Pruning + Early stopping для ускорения в 2-3 раза
# - Используем все 5 folds для оценки (качество не теряется)

import gc
from pathlib import Path

# Очищаем память перед началом
gc.collect()

# Запускаем по одному split_mode за раз
all_results = []

for split_mode in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    print(f"\n{'='*60}")
    print(f"Processing split_mode: {split_mode}")
    print(f"{'='*60}\n")
    
    try:
        results_df = screen_lgbm_optuna(
            df,
            apt_cfgs,
            mol_cfgs,
            split_modes=(split_mode,),  # По одному за раз
            n_trials=10,  # Количество trials
            metric="roc_auc",
            splits_dir="../dataset/splits",
            use_cached_embeddings=True,  # Используем кэш эмбеддингов
            checkpoint_dir="notebooks/checkpoints",  # Директория для чекпоинтов
            results_file=f"notebooks/checkpoints/results_{split_mode}.csv",  # Онлайн сохранение
            resume=True,  # Продолжить с чекпоинта если есть
            use_pruning=True,  # Pruning для ускорения (~30-50%)
        )
        all_results.append(results_df)
        
        # Сохраняем финальные результаты
        results_df.to_csv(f'lgbm_results_{split_mode}.csv', index=False)
        print(f"✓ Saved final results for {split_mode}")
        
    except Exception as e:
        print(f"✗ Error processing {split_mode}: {e}")
        import traceback
        traceback.print_exc()
        # Пытаемся загрузить частичные результаты
        checkpoint_file = f"notebooks/checkpoints/results_{split_mode}.csv"
        if Path(checkpoint_file).exists():
            print(f"⚠ Загружаем частичные результаты из {checkpoint_file}")
            try:
                partial_df = pd.read_csv(checkpoint_file)
                all_results.append(partial_df)
            except:
                pass
    
    # Очищаем память после каждого split_mode
    gc.collect()
    print(f"Memory cleared after {split_mode}\n")

# Объединяем все результаты
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv('lgbm_results_screening_complete.csv', index=False)
    print("✓ All results saved to lgbm_results_screening_complete.csv")
else:
    print("✗ No results collected")



Processing split_mode: stratified

✓ Загружено 35 существующих результатов. Продолжаем с 35 комбинаций.


Apt encoders (stratified):   0%|          | 0/5 [00:00<?, ?it/s]

  ⏭ Пропускаем OneHot × MorganFP (уже выполнено)
  ⏭ Пропускаем OneHot × MACCS (уже выполнено)
  ⏭ Пропускаем OneHot × PhysChem (уже выполнено)
  ⏭ Пропускаем OneHot × ChemBERTa (уже выполнено)
  ⏭ Пропускаем OneHot × UniMol (уже выполнено)


Apt encoders (stratified):  20%|██        | 1/5 [00:00<00:02,  1.39it/s]

  ⏭ Пропускаем OneHot × MolFormer (уже выполнено)
  ⏭ Пропускаем OneHot × CLAMP (уже выполнено)
  ⏭ Пропускаем Kmer3 × MorganFP (уже выполнено)
  ⏭ Пропускаем Kmer3 × MACCS (уже выполнено)
  ⏭ Пропускаем Kmer3 × PhysChem (уже выполнено)
  ⏭ Пропускаем Kmer3 × ChemBERTa (уже выполнено)
  ⏭ Пропускаем Kmer3 × UniMol (уже выполнено)
  ⏭ Пропускаем Kmer3 × MolFormer (уже выполнено)
  ⏭ Пропускаем Kmer3 × CLAMP (уже выполнено)


Apt encoders (stratified):  40%|████      | 2/5 [00:01<00:01,  2.13it/s]

... (27 outputs left out)

Apt encoders (disjoint_aptamer):  80%|████████  | 4/5 [00:01<00:00,  3.32it/s]

  ⏭ Пропускаем DNABERT2 × MorganFP (уже выполнено)
  ⏭ Пропускаем DNABERT2 × MACCS (уже выполнено)
  ⏭ Пропускаем DNABERT2 × PhysChem (уже выполнено)
  ⏭ Пропускаем DNABERT2 × ChemBERTa (уже выполнено)
  ⏭ Пропускаем DNABERT2 × UniMol (уже выполнено)
  ⏭ Пропускаем DNABERT2 × MolFormer (уже выполнено)
  ⏭ Пропускаем DNABERT2 × CLAMP (уже выполнено)


Apt encoders (disjoint_aptamer): 100%|██████████| 5/5 [00:01<00:00,  3.30it/s]

✓ Saved final results for disjoint_aptamer
Memory cleared after disjoint_aptamer

✓ All results saved to lgbm_results_screening_complete.csv


## 4. Regression Model on Best Encoder Combination

Based on the classification results, the best encoder combination is **Kmer4 + PhysChem** (MCC=0.7796).
Now we'll train a regression model to predict pKd values using only samples with valid pKd measurements.
